# Nanopore analysis (from the PRJEB88111 dataset)

In [1]:
import polars as pl
import matplotlib.pyplot as plt

## Read in summary of mapping of nanopore to reference genomes

In [2]:
depth_df = (
    pl.read_csv('../outputs.PRJEB88111.nanopore/bams/*.x.all-core.depth.csv')
)
depth_df.head(1)

metag,gene,len,hits,breadth,depth_all,depth_cov
str,str,i64,i64,f64,f64,f64
"""ERR15909852""","""FR882202.1""",72839,5938,0.081522,0.088785,1.089087


## Read in table connecting contigs to species

In [3]:
contigs_species_df = pl.read_csv('../outputs.PRJEB88111.nanopore/contigs_to_species.csv')
contigs_species_df.head(1)

contig,species,acc
str,str,str
"""UREV01000001.1""","""s__Cryptobacteroides sp9005469…","""GCA_900546925.1"""


## Combine!

In [4]:
join_df = (
    depth_df
    .join(contigs_species_df, left_on='gene', right_on='contig', how='inner')
    .with_columns(
        weighted_depth=(pl.col('len') * pl.col('depth_all')),
        weighted_breadth=(pl.col('len') * pl.col('breadth')),
    )
)
join_df.head(1)

metag,gene,len,hits,breadth,depth_all,depth_cov,species,acc,weighted_depth,weighted_breadth
str,str,i64,i64,f64,f64,f64,str,str,f64,f64
"""ERR15909852""","""FR882202.1""",72839,5938,0.081522,0.088785,1.089087,"""s__Cryptobacteroides sp0004349…","""GCA_000434935.1""",6467.0,5938.0


## Summarize by metag/species

In [5]:
sum_df = join_df.group_by(['metag', 'species']).agg(
    depth_all=pl.col('weighted_depth').sum() / pl.col('len').sum(),
    breadth=pl.col('weighted_breadth').sum() / pl.col('len').sum(),
)
sum_df.sort('breadth')

metag,species,depth_all,breadth
str,str,f64,f64
"""ERR15942132""","""s__Cryptobacteroides sp9005469…",0.0,0.0
"""ERR15941754""","""s__Cryptobacteroides sp9005469…",0.0,0.0
"""ERR15942135""","""s__Cryptobacteroides sp0004326…",0.0,0.0
"""ERR15942245""","""s__Cryptobacteroides sp9005469…",0.0,0.0
"""ERR15942238""","""s__Cryptobacteroides sp0004326…",0.0,0.0
…,…,…,…
"""ERR15935028""","""s__JAFBIX01 sp021531895""",137.764068,0.992922
"""ERR15934940""","""s__Bariatricus sp004560705""",108.242687,0.99321
"""ERR15934725""","""s__Bariatricus sp004560705""",102.132863,0.993625


## Read in the manifest information for the sketches

In [6]:
mf_df = (
    pl.read_csv('../outputs.PRJEB88111.nanopore/PRJEB88111.mf.csv', skip_lines=1)
    .select(["name", "n_hashes"])
)

## Add it in to the summary...

In [7]:
sum2_df = sum_df.join(mf_df, left_on='metag', right_on='name', how='inner')
sum2_df

metag,species,depth_all,breadth,n_hashes
str,str,f64,f64,i64
"""ERR15909856""","""s__Prevotella sp002251295""",0.138455,0.094686,96666
"""ERR15909856""","""s__Prevotella sp002251295""",0.138455,0.094686,101637
"""ERR15909856""","""s__Prevotella sp002251295""",0.138455,0.094686,109515
"""ERR15942235""","""s__Prevotella sp002251295""",0.114299,0.001396,115408
"""ERR15942235""","""s__Prevotella sp002251295""",0.114299,0.001396,156558
…,…,…,…,…
"""ERR15942141""","""s__Mogibacterium_A kristiansen…",0.814743,0.003886,139839
"""ERR15942141""","""s__Mogibacterium_A kristiansen…",0.814743,0.003886,206060
"""ERR15942221""","""s__Cryptobacteroides sp0340892…",0.014325,0.000858,71962


## ...and now filter for the bigger sketches only, representing the deeper sequenced nanopore runs.

See analysis at bottom for motivation!

In [8]:
filt_df = sum2_df.filter(pl.col('n_hashes') > 4e6)
filt_df

metag,species,depth_all,breadth,n_hashes
str,str,f64,f64,i64
"""ERR15934940""","""s__Cryptobacteroides sp0004326…",5.256308,0.890695,5241832
"""ERR15934940""","""s__Cryptobacteroides sp0004326…",5.256308,0.890695,6757251
"""ERR15934940""","""s__Cryptobacteroides sp0004326…",5.256308,0.890695,9212112
"""ERR15935028""","""s__Cryptobacteroides sp9005469…",8.122636,0.969058,4745917
"""ERR15935028""","""s__Cryptobacteroides sp9005469…",8.122636,0.969058,5871776
…,…,…,…,…
"""ERR15934513""","""s__UBA2868 sp004552595""",51.646273,0.333482,7841501
"""ERR15934513""","""s__UBA2868 sp004552595""",51.646273,0.333482,11162227
"""ERR15935218""","""s__Fimisoma sp002320005""",12.668152,0.859564,4607043


## Summarize by species (over metagenomes), now.

In [9]:
by_species = filt_df.group_by('species').agg(
    min_breadth=pl.col('breadth').min().round(3),
    med_breadth=pl.col('breadth').median().round(3),
    max_breadth=pl.col('breadth').max().round(3),
    min_depth=pl.col('depth_all').min().round(2),
    med_depth=pl.col('depth_all').median().round(1),
    max_depth=pl.col('depth_all').max().round(1),
)

In [10]:
with pl.Config(tbl_rows=-1):
    print(by_species.sort('min_breadth'))

shape: (16, 7)
┌────────────────────┬─────────────┬─────────────┬─────────────┬───────────┬───────────┬───────────┐
│ species            ┆ min_breadth ┆ med_breadth ┆ max_breadth ┆ min_depth ┆ med_depth ┆ max_depth │
│ ---                ┆ ---         ┆ ---         ┆ ---         ┆ ---       ┆ ---       ┆ ---       │
│ str                ┆ f64         ┆ f64         ┆ f64         ┆ f64       ┆ f64       ┆ f64       │
╞════════════════════╪═════════════╪═════════════╪═════════════╪═══════════╪═══════════╪═══════════╡
│ s__UBA2868         ┆ 0.33        ┆ 0.798       ┆ 0.929       ┆ 5.66      ┆ 19.6      ┆ 52.3      │
│ sp004552595        ┆             ┆             ┆             ┆           ┆           ┆           │
│ s__Floccifex porci ┆ 0.45        ┆ 0.944       ┆ 0.97        ┆ 11.98     ┆ 31.4      ┆ 67.1      │
│ s__Mogibacterium_A ┆ 0.484       ┆ 0.963       ┆ 0.991       ┆ 8.89      ┆ 24.9      ┆ 80.4      │
│ kristiansen…       ┆             ┆             ┆             ┆           ┆

## APPENDIX: analyze the manysearch data

In [11]:
orig_df = (
    pl.scan_csv('../outputs.PRJEB88111.nanopore/PRJEB88111.x.mags+gtdb.manysearch.csv')
    .with_columns(species_name=pl.col('query_name').str.split(' ').list.slice(1, 2).list.join( " "))
).collect()

In [12]:
CORE_NAMES = [ x.strip() for x in open('../inputs.branchwater/names.list') ]
CORE_NAMES

['s__Bariatricus sp004560705',
 's__Colivicinus sp002299675',
 's__Cryptobacteroides sp000432655',
 's__Cryptobacteroides sp000434935',
 's__Cryptobacteroides sp034089285',
 's__Cryptobacteroides sp900546925',
 's__Fimisoma sp002320005',
 's__Floccifex porci',
 's__JAFBIX01 sp021531895',
 's__Lactobacillus amylovorus',
 's__Mogibacterium_A kristiansenii',
 's__Ornithospirochaeta sp022785155',
 's__Prevotella sp000434975',
 's__Prevotella sp002251295',
 's__Sodaliphilus sp004557565',
 's__UBA2868 sp004552595']

In [13]:
df = (orig_df
      .filter(pl.col('intersect_hashes') >= 20)
      .filter(pl.col('species_name').is_in(CORE_NAMES))
     )                                       

In [14]:
by_acc = df.group_by('match_name').agg(
    n_core=pl.len(),
    med_intersect=pl.col('intersect_hashes').median(),
)
with pl.Config(tbl_rows=-1):
    print(by_acc.sort('n_core'))

shape: (59, 3)
┌─────────────┬────────┬───────────────┐
│ match_name  ┆ n_core ┆ med_intersect │
│ ---         ┆ ---    ┆ ---           │
│ str         ┆ u32    ┆ f64           │
╞═════════════╪════════╪═══════════════╡
│ ERR15909890 ┆ 6      ┆ 46.0          │
│ ERR15909878 ┆ 7      ┆ 64.0          │
│ ERR15909855 ┆ 8      ┆ 31.5          │
│ ERR15909872 ┆ 8      ┆ 49.0          │
│ ERR15909869 ┆ 8      ┆ 70.5          │
│ ERR15909853 ┆ 8      ┆ 54.0          │
│ ERR15909856 ┆ 8      ┆ 55.0          │
│ ERR15909888 ┆ 9      ┆ 60.0          │
│ ERR15909889 ┆ 9      ┆ 46.0          │
│ ERR15909852 ┆ 9      ┆ 97.0          │
│ ERR15909877 ┆ 9      ┆ 76.0          │
│ ERR15909859 ┆ 9      ┆ 50.0          │
│ ERR15909887 ┆ 9      ┆ 61.0          │
│ ERR15909879 ┆ 10     ┆ 63.5          │
│ ERR15909882 ┆ 10     ┆ 88.0          │
│ ERR15909885 ┆ 10     ┆ 41.0          │
│ ERR15909892 ┆ 10     ┆ 33.5          │
│ ERR15909874 ┆ 10     ┆ 63.5          │
│ ERR15909873 ┆ 11     ┆ 66.0          │
│

In [15]:
manysearch_join_df = (
    by_acc
    .join(mf_df, left_on='match_name', right_on='name', how='inner')
    .select(['match_name', 'n_core', 'n_hashes', 'med_intersect']) 
)
with pl.Config(tbl_rows=-1):
    print(manysearch_join_df.sort('n_hashes'))

shape: (177, 4)
┌─────────────┬────────┬──────────┬───────────────┐
│ match_name  ┆ n_core ┆ n_hashes ┆ med_intersect │
│ ---         ┆ ---    ┆ ---      ┆ ---           │
│ str         ┆ u32    ┆ i64      ┆ f64           │
╞═════════════╪════════╪══════════╪═══════════════╡
│ ERR15909855 ┆ 8      ┆ 63791    ┆ 31.5          │
│ ERR15909855 ┆ 8      ┆ 65550    ┆ 31.5          │
│ ERR15909855 ┆ 8      ┆ 67653    ┆ 31.5          │
│ ERR15909890 ┆ 6      ┆ 74127    ┆ 46.0          │
│ ERR15909890 ┆ 6      ┆ 76655    ┆ 46.0          │
│ ERR15909888 ┆ 9      ┆ 80173    ┆ 60.0          │
│ ERR15909890 ┆ 6      ┆ 80687    ┆ 46.0          │
│ ERR15909888 ┆ 9      ┆ 82205    ┆ 60.0          │
│ ERR15909878 ┆ 7      ┆ 82496    ┆ 64.0          │
│ ERR15909853 ┆ 8      ┆ 83559    ┆ 54.0          │
│ ERR15909858 ┆ 11     ┆ 83948    ┆ 52.0          │
│ ERR15909878 ┆ 7      ┆ 84293    ┆ 64.0          │
│ ERR15909888 ┆ 9      ┆ 85657    ┆ 60.0          │
│ ERR15909853 ┆ 8      ┆ 85907    ┆ 54.0        

### ...and examine table above to find cutoff where the big datasets start <=> all 16 species.